In [1]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision
from torchvision.transforms import transforms as T
from torch.utils.data import DataLoader, Subset

In [2]:
t = T.ToTensor()
trd = DataLoader(Subset(torchvision.datasets.CIFAR10('./data',True,download=True,transform=t),range(200)),batch_size=10, shuffle=True)
ted = DataLoader(Subset(torchvision.datasets.CIFAR10('./data',False,download=True, transform=t),range(50)),batch_size=10,shuffle=False)

100.0%


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified


In [5]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(4096,512), nn.ReLU(), nn.Dropout(0.5), nn.Linear(512,10)
        )
    def forward(self,x):
        return self.net(x)


In [10]:
def train(model,opt,crit,epochs):
    for e in range(epochs):
        model.train()
        rl,tc,tn=0,0,0
        for x,y in trd:
            opt.zero_grad()
            o = model(x)
            l = crit(o,y)
            l.backward()
            opt.step()
            rl += l.item()
            tc += (o.argmax(1)==y).sum().item()
            tn += len(y)
        
        tl,c,n=0,0,0
        model.eval()
        with torch.no_grad():
            for x,y in ted:
                o = model(x)
                l = crit(o,y)
                tl += l.item()
                c += (o.argmax(1)==y).sum().item()
                n += len(y)
        print(f'E:{e+1} | Train Loss : {rl/len(trd):.2f} Train Accuracy : {tc/tn*100:.2f}% | Test Loss : {tl/len(ted):.2f} Accuracy : {c/n*100:.2f}')

model = CNN()
opt = optim.Adam(model.parameters(),1e-3)
crit = nn.CrossEntropyLoss()
train(model,opt,crit,30)

E:1 | Train Loss : 3.40 Train Accuracy : 14.00% | Test Loss : 2.23 Accuracy : 18.00
E:2 | Train Loss : 2.43 Train Accuracy : 27.00% | Test Loss : 2.07 Accuracy : 20.00
E:3 | Train Loss : 1.86 Train Accuracy : 35.50% | Test Loss : 2.15 Accuracy : 16.00
E:4 | Train Loss : 1.60 Train Accuracy : 47.00% | Test Loss : 1.86 Accuracy : 26.00
E:5 | Train Loss : 1.46 Train Accuracy : 51.50% | Test Loss : 2.03 Accuracy : 28.00
E:6 | Train Loss : 1.36 Train Accuracy : 56.50% | Test Loss : 2.16 Accuracy : 26.00
E:7 | Train Loss : 1.21 Train Accuracy : 58.50% | Test Loss : 1.96 Accuracy : 36.00
E:8 | Train Loss : 1.00 Train Accuracy : 65.00% | Test Loss : 2.32 Accuracy : 18.00
E:9 | Train Loss : 1.02 Train Accuracy : 69.00% | Test Loss : 2.39 Accuracy : 22.00
E:10 | Train Loss : 0.85 Train Accuracy : 70.50% | Test Loss : 2.10 Accuracy : 26.00
E:11 | Train Loss : 0.80 Train Accuracy : 73.50% | Test Loss : 2.16 Accuracy : 24.00
E:12 | Train Loss : 0.69 Train Accuracy : 78.50% | Test Loss : 2.22 Accura